In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup

import zipfile

import pandas as pd

from pandas import ExcelWriter

import datetime

from selenium import webdriver

from time import sleep

import re

import os

from googletrans import Translator
from selenium.webdriver.chrome.service import Service as ChromeService

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.support.ui import Select 


In [11]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'BR BCB' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)




Running BR BCB Web Scraping Tool v.1.3


In [ ]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

		 'profile.default_content_setting_values.automatic_downloads': 1}

chromeOptions.add_experimental_option("prefs",prefs)


driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={
    regulatorName + ' 1': 'https://www.bcb.gov.br/estabilidadefinanceira/relacao_instituicoes_funcionamento', 

	# regulatorName + ' 2': 'https://www.bcb.gov.br/estabilidadefinanceira/relacao_instituicoes_funcionamento', 

	# regulatorName + ' 3': 'https://www.bcb.gov.br/estabilidadefinanceira/relacao_instituicoes_funcionamento', 

	# regulatorName + ' 4': 'https://www.bcb.gov.br/estabilidadefinanceira/relacao_instituicoes_funcionamento', 

	# regulatorName + ' 5': 'https://www.bcb.gov.br/estabilidadefinanceira/relacao_instituicoes_funcionamento', 
   
   
	# BR BCB 8 is chart 10 - Representative of Foreign institutions in the Country
	regulatorName + ' 8': 'https://www.bcb.gov.br/en/statistics/evolutionmonthnfs'
   }

Typology={

		regulatorName + ' 1': 'Conglomerados',
		regulatorName + ' 2': 'Bancos comerciais, múltiplos e Caixa Econômica',
		regulatorName + ' 3': 'Cooperativas de crédito',
		regulatorName + ' 4': 'Bancos de Investimento, Bancos de Desenvolvimento, Sociedades Corretoras de TVM e Câmbio, Sociedades Distribuidoras de TVM, Sociedades de Crédito, Financiamento e Investimento, Sociedades de Crédito Imobiliário e APE, Sociedades de Arrendamento Mercantil, Sociedades de Investimento, Sociedades de Crédito ao Microempreendedor, Agências de Fomento, Companhias Hipotecárias e Instituições de Pagamento',
		regulatorName + ' 5': 'Administradoras de consórcios',
		regulatorName + ' 8': 'Supervised institutions - Foreign Institutions in Brazil',
        }

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



ISO= {"": "", 'NAN': '', 'OTHER': '', "AFGHANISTAN": "AF", "ÅLAND ISLANDS": "AX", "ALBANIA": "AL", "ALGERIA": "DZ", "AMERICAN SAMOA": "AS", "ANDORRA": "AD", "ANGOLA": "AO", "ANGUILLA": "AI", "ANTARCTICA": "AQ", "ANTIGUA AND BARBUDA": "AG", "ARGENTINA": "AR", "ARMENIA": "AM", "ARUBA": "AW", "AUSTRALIA": "AU", "AUSTRIA": "AT", "AZERBAIJAN": "AZ", "BAHAMAS, THE": "BS", "BAHRAIN": "BH", "BANGLADESH": "BD", "BARBADOS": "BB", "BELARUS": "BY", "BELGIUM": "BE", "BELIZE": "BZ", "BENIN": "BJ", "BERMUDA": "BM", "BHUTAN": "BT", "BOLIVIA": "BO", "BONAIRE, SINT EUSTATIUS AND SABA": "BQ", "BOSNIA AND HERZEGOVINA": "BA", "BOTSWANA": "BW", "BOUVET ISLAND": "BV", "BRAZIL": "BR", "BRITISH INDIAN OCEAN TERRITORY": "IO", "BRUNEI": "BN", "BULGARIA": "BG", "BURKINA FASO": "BF", "BURUNDI": "BI", "CABO VERDE": "CV", "CAMBODIA": "KH", "CAMEROON, UNITED REPUBLIC OF": "CM", "CANADA": "CA", "CAYMAN ISLANDS": "KY", "CENTRAL AFRICAN REPUBLIC": "CF", "CHAD": "TD", "CHILE": "CL", "CHINA, PEOPLES REPUBLIC OF": "CN","CHINA": "CN", "CHRISTMAS ISLAND": "CX", "COCOS (KEELING) ISLANDS": "CC", "COLOMBIA": "CO", "COMOROS": "KM", "CONGO": "CG", "CONGO, DEMOCRATIC REPUBLIC OF THE": "CD", "COOK ISLANDS": "CK", "COSTA RICA": "CR", "CÔTE D'IVOIRE": "CI", "CROATIA": "HR", "CUBA": "CU", "CURACAO, BONAIRE, SABA, ST. MARTIN & ST.": "CW", "CYPRUS": "CY", "CZECH REPUBLIC": "CZ", "DENMARK": "DK", "DJIBOUTI": "DJ", "DOMINICA": "DM", "DOMINICAN REPUBLIC": "DO", "ECUADOR": "EC", "EGYPT": "EG", "EL SALVADOR": "SV", "EQUATORIAL GUINEA": "GQ", "ERITREA": "ER", "ESTONIA": "EE", "ESWATINI": "SZ", "ETHIOPIA": "ET", "FALKLAND ISLANDS (MALVINAS)": "FK", "FAROE ISLANDS": "FO", "FIJI": "FJ", "FINLAND": "FI", "FRANCE": "FR", "FRENCH GUIANA": "GF", "FRENCH POLYNESIA": "PF", "FRENCH SOUTHERN TERRITORIES": "TF", "GABON": "GA", "GAMBIA": "GM", "GEORGIA": "GE", 'GEORGIA/GRUZINSKAYA': 'GE', "GERMANY": "DE", "GHANA": "GH", "GIBRALTAR": "GI", "GREECE": "GR", "GREENLAND": "GL", "GRENADA": "GD", "GUADELOUPE": "GP", "GUAM": "GU", "GUATEMALA": "GT", "GUERNSEY": "GG", "GUINEA": "GN", "GUINEA-BISSAU": "GW", "GUYANA": "GY", "HAITI": "HT", "HEARD ISLAND AND MCDONALD ISLANDS": "HM", "HOLY SEE": "VA", "HONDURAS": "HN", "HONG KONG": "HK", "HUNGARY": "HU", "ICELAND": "IS", "INDIA": "IN", "INDONESIA": "ID", "IRAN": "IR", "IRAQ": "IQ", "IRELAND": "IE", "ISLE OF MAN": "IM", "ISRAEL": "IL", "ITALY": "IT", "JAMAICA": "JM", "JAPAN": "JP", "JERSEY": "JE", "JORDAN": "JO", "KAZAKHSTAN": "KZ", "KENYA": "KE", "KIRIBATI": "KI", """KOREA (DEMOCRATIC PEOPLE'S REPUBLIC OF)""": "KP", "KOREA, SOUTH": "KR", "SOUTH KOREA": "KR", "KUWAIT": "KW", "KYRGYZSTAN": "KG", "LAO PEOPLE'S DEMOCRATIC REPUBLIC": "LA", "LATVIA": "LV", "LEBANON": "LB", "LESOTHO": "LS", "LIBERIA": "LR", "LIBYA": "LY", "LIECHTENSTEIN": "LI", "LITHUANIA": "LT", "LUXEMBOURG": "LU", "MACAU": "MO", "MADAGASCAR": "MG", "MALAWI": "MW", "MALAYSIA": "MY", "MALDIVES": "MV", "MALI": "ML", "MALTA": "MT", "MARSHALL ISLANDS": "MH", "MARTINIQUE": "MQ", "MAURITANIA": "MR", "MAURITIUS": "MU", "MAYOTTE": "YT", "MEXICO": "MX", "FEDERATED STATES OF MICRONESIA": "FM", "MOLDOVA, REPUBLIC OF": "MD", "MONACO": "MC", "MONGOLIA": "MN", "MONTENEGRO": "ME", "MONTSERRAT": "MS", "MOROCCO": "MA", "MOZAMBIQUE": "MZ", "MYANMAR": "MM", "NAMIBIA": "NA", "NAURU": "NR", "NEPAL": "NP", "NETHERLANDS": "NL", "NEW CALEDONIA": "NC", "NEW ZEALAND": "NZ", "NICARAGUA": "NI", "NIGER": "NE", "NIGERIA": "NG", "NIUE": "NU", "NORFOLK ISLAND": "NF", "NORTH MACEDONIA": "MK", "NORTHERN MARIANA ISLANDS": "MP", "NORWAY": "NO", "OMAN": "OM", "PAKISTAN": "PK", "PALAU": "PW", "PALESTINE, STATE OF": "PS", "PANAMA": "PA", "PAPUA NEW GUINEA": "PG", "PARAGUAY": "PY", "PERU": "PE", "PHILIPPINES": "PH", "PITCAIRN": "PN", "POLAND": "PL", "PORTUGAL": "PT", "PUERTO RICO": "PR", "QATAR": "QA", "RÉUNION": "RE", "ROMANIA": "RO", "RUSSIA": "RU", "RWANDA": "RW", "SAINT BARTHÉLEMY": "BL", "SAINT HELENA, ASCENSION AND TRISTAN DA CUNHA": "SH", "SAINT KITTS AND NEVIS": "KN", "SAINT LUCIA": "LC", "SAINT MARTIN (FRENCH PART)": "MF", "SAINT PIERRE AND MIQUELON": "PM", "SAINT VINCENT AND THE GRENADINES": "VC", "SAMOA": "WS", "SAN MARINO": "SM", "SAO TOME AND PRINCIPE": "ST", "SAUDI ARABIA": "SA", "SENEGAL": "SN", "SERBIA": "RS", "SEYCHELLES": "SC", "SIERRA LEONE": "SL", "SINGAPORE": "SG", "SINT MAARTEN (DUTCH PART)": "SX", "SLOVAKIA": "SK", "SLOVAK REPUBLIC": "SK", "SLOVENIA": "SI", "SOLOMON ISLANDS": "SB", "SOMALIA": "SO", "SOUTH AFRICA": "ZA", "SOUTH GEORGIA AND THE SOUTH SANDWICH ISLANDS": "GS", "SOUTH SUDAN": "SS", "SPAIN": "ES", "SRI LANKA": "LK", "SUDAN": "SD", "SURINAME": "SR", "SVALBARD AND JAN MAYEN": "SJ", "SWEDEN": "SE", "SWITZERLAND": "CH", "SYRIAN ARAB REPUBLIC": "SY", "TAIWAN": "TW",'TAIWAN,  REPUBLIC OF CHINA': 'TW' ,"TAJIKISTAN": "TJ", "TANZANIA, UNITED REPUBLIC OF": "TZ", "THAILAND": "TH", "TIMOR-LESTE": "TL", "TOGO": "TG", "TOKELAU": "TK", "TONGA": "TO", "TRINIDAD AND TOBAGO": "TT", "TUNISIA": "TN", "TURKEY": "TR", "TURKMENISTAN": "TM", "TURKS & CAICOS ISLANDS": "TC", "TUVALU": "TV", "UGANDA": "UG", "UKRAINE": "UA", "UNITED ARAB EMIRATES": "AE", "UNITED KINGDOM OF GREAT BRITAIN AND NORTHERN IRELAND": "GB", "UNITED STATES": "US", 'USA': 'US', "UNITED STATES MINOR OUTLYING ISLANDS": "UM", "URUGUAY": "UY", "UZBEKISTAN": "UZ", "VANUATU": "VU", "VENEZUELA": "VE", "VIETNAM": "VN", "BRITISH VIRGIN ISLANDS": "VG", "VIRGIN ISLANDS OF THE U.S.": "VI", "WALLIS AND FUTUNA": "WF", "WESTERN SAHARA": "EH", "YEMEN": "YE", "ZAMBIA": "ZM", "ZIMBABWE": "ZW", "ENGLAND": "GB", "UNITED KINGDOM": "GB", "UNITED KINGDOM (OTHER)": "GB", "FRANCE (OTHER)": "FR", "WALES": "GB", "CONGO (KINSHASA)": "CD", "CONGO (BRAZZAVILLE)": "CD", "SCOTLAND": "GB", "ITALY (OTHER)": "IT", "INDONESIA (OTHER)": "ID", "INDIA (OTHER)": "IN", "MOROCCO (OTHER)": "MA", "NEW ZEALAND (OTHER)": "NZ", "SWITZERLAND (OTHER)": "CH", "MALAYSIA (OTHER)": "MY", "NETHERLANDS ANTILLES": "AN", "TRINIDAD & TOBAGO (OTHER)": "TT", "CHANNEL ISLANDS": "GB", "UNITED ARAB EMIRATES (OTHER)": "AE", "DENMARK (OTHER)": "DK", "COMORO ISLANDS": "KM", "MACEDONIA (FORMER YUGOSLAV REPUBLIC OF)": "MK", "SERBIA AND MONTENEGRO(FORMER YUGOSLAVIA)": "CS", "TRINIDAD": "TT", "ETHIOPIA (OTHER)": "ET", "IVORY COAST": "CI", "DUBAI": "AE", "BRITISH WEST INDIES (OTHER)": "VG", "SWAZILAND": "SZ", 'UNITED KINGDOM  (OTHER)': 'GB','BAHAMAS':'BS'}

STATES={'': '', 'NaN': '', 'AC': 'Rio Branco', 'AL': 'Maceió', 'AP': 'Macapá', 'AM': 'Manaus', 'BA': 'Salvador', 'CE': 'Fortaleza', 'DF': 'Brasília', 'ES': 'Vitória', 'GO': 'Goiânia', 'MA': 'São Luís', 'MT': 'Cuiabá', 'MS': 'Campo Grande', 'MG': 'Belo Horizonte', 'PA': 'Belém', 'PB': 'João Pessoa', 'PR': 'Curitiba', 'PE': 'Recife', 'PI': 'Teresina', 'RJ': 'Rio de Janeiro', 'RN': 'Natal', 'RS': 'Porto Alegre', 'RO': 'Porto Velho', 'RR': 'Boa Vista', 'SC': 'Florianópolis', 'SP': 'São Paulo', 'SE': 'Aracaju', 'TO': 'Palmas'}

processdate=now.strftime('%Y-%m-%d')


In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict





def click_on_cookies(web_driver):

    try:

        try:

            web_driver.find_element(By.XPATH,f'/html/body/app-root/bcb-cookies/div/div/div/div/button[2]').click()

        except:

            web_driver.find_element(By.XPATH,f'/html/body/app-root/bcb-cookies/div/div/div/div/button[2]').click()

        

        print("[INFO] : - Click cookies")



    except Exception as err:

        print('[ERROR] : - Failed to click "Prosseguir" button on the cookies banner:', err)





def check_dowload_files(tempfolder, fileType ):



    for time in range(10):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )
    
translator = Translator()
def translate_text(text):
    sleep(2)
    return translator.translate(text, src='pt', dest='en').text


In [6]:
# %%

class CustomException(Exception):
	pass

#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
	print(f'Working with list {reg}')
	driver.get(regdict[reg])
	sleep(10)
	soup = BeautifulSoup(driver.page_source, 'html.parser')  
	sleep(5)
	if soup.find('div', {'class':'msg-cookies card border-0'}):
		selenium_element = driver.find_element(By.XPATH, "//div[@class='msg-cookies card border-0']//div[@class='text-center']//button[last()]")
		driver.execute_script("arguments[0].click();", selenium_element)
		print('----- Click Cookie-----')
		driver.get(regdict[reg])
	if reg != 'BR BCB 8':
		#driver.refresh()
		sleep(10)
		soup = BeautifulSoup(driver.page_source, 'html.parser')
		sleep(5)
		main_container = soup.find('div',class_='container main')
		table_link = main_container.find('iframe')['src']
		sleep(5)
		driver.get(table_link)
		sleep(5)
		soup_table = BeautifulSoup(driver.page_source, 'html.parser')
		sleep(5)  
		table = soup_table.find('table')
		tbody = table.find('tbody')
		trs = tbody.find_all('tr')
		for index,tr in enumerate(trs):
			options = tr.find_all('td')[1].find('select').find_all('option')
			if index == 0:
				keyword1 = 'conglomerado'
				keyword2 = 'conglomerado'
			elif index ==1:
				keyword1 = 'banco'
				keyword2 = 'banco'
			elif index ==2:
				keyword1 = 'cooperativa'
				keyword2 = 'cooperativa'
			elif index ==3:
				keyword1 = 'sociedade'
				keyword2 = 'sociedade'
			elif index ==4:
				keyword1 = 'admConsorcio'
				keyword2 = 'consorcio'
			print(f'---- Work with {keyword1}')
			reg = regulatorName + ' ' + str(int(index)+1) 
			drop=Select(driver.find_element(By.XPATH, f'//*[@id="{keyword1}"]'))
			button = driver.find_element(By.XPATH, f"//button[@ng-click=\"baixarArquivo('{keyword2}');\"]")
			sleep(5)
 
			drop.select_by_index(1)
			driver.execute_script("arguments[0].click();", button)
			sleep(5)
			try:
				file_list = os.listdir(tempfolder)
				if file_list:
					file = file_list[0]
					print('This file downloads successfully!')
					filePath = os.path.join(tempfolder, file)
				else:
					file_list = os.listdir(tempfolder)
					file = os.listdir(tempfolder)[0]
					filePath = os.path.join(tempfolder, file)
			except Exception as e:
				print(f'An error occurred: {e}')
			try:
				sleep(5)
				with zipfile.ZipFile(filePath, 'r') as zip_ref:
					sleep(5)
					zip_ref.extractall(tempfolder)
					sleep(5)
				# print(f"[INFO] : - Zip extraction file = {zip_ref.filelist[0].filename}")
			except Exception as e:
				print(f'An error occurred: {e}')
    
			sleep(35)
			extracted_files = os.listdir(tempfolder)
			xlsx_files = [file for file in extracted_files if file.endswith('.xlsx')]
			sleep(35)
			# Check if any .xlsx files were found
			try:
				if xlsx_files:
					print(f"[INFO] : - Found .xlsx file(s): {xlsx_files}")
					sleep(35)
					filePath = os.path.join(tempfolder, xlsx_files[0])
				else:
					extracted_files = os.listdir(tempfolder)
					xlsx_files = [file for file in extracted_files if file.endswith('.xlsx')]
					filePath = os.path.join(tempfolder, xlsx_files[0])
					print("[ERROR] : - No .xlsx files found first time, try again")

			except Exception as e:
				extracted_files = os.listdir(tempfolder)
				xlsx_files = [file for file in extracted_files if file.endswith('.xlsx')]
				sleep(35)
				if xlsx_files:
					print(f"[INFO] : - Found .xlsx file(s): {xlsx_files}")
					sleep(35)
					filePath = os.path.join(tempfolder, xlsx_files[0])
				else:
					print(f'Error retrieving file: {e}')


			data_Excel=pd.read_excel(filePath)
			if reg == regulatorName + ' 1':
				data_Excel=data_Excel.fillna('')
				data_Excel = data_Excel[~((data_Excel['Unnamed: 1'].str.strip() == '') & (data_Excel['Unnamed: 2'].str.strip() == ''))]
				data_Excel.reset_index(drop=True, inplace=True)
				data_Excel.columns = data_Excel.iloc[0]
				data_Excel.columns = list(map(str.strip, data_Excel))
				data_Excel = data_Excel[1:]
				sleep(3)
				for mother_name,name,cngj,date in zip(data_Excel['NOME DO CONGLOMERADO'],
													data_Excel['NOME DO PARTICIPANTE'],
													data_Excel['CNPJ PARTICIPANTE'],
													data_Excel['DATA INÍCIO']):

					
					sqldict['Name'].append(name.strip())
					sqldict['InternalID_1'].append(str(cngj).strip())
					sqldict['InternalID_1_type'].append('CNPJ PARTICIPANTE')
					sqldict['ListProcessDate'].append(processdate)
					sqldict['Name - Mother Company'].append(mother_name.strip())
					sqldict['RegulationDate'].append(date.date())
					sqldict['ListName'].append(Typology[reg])
					sqldict['RegCtry'].append(reg.split(' ')[0]) 
					sqldict['RegCode'].append(reg.split(' ')[1])
					sqldict['ListCode'].append(reg.split(' ')[-1])          
					sqldict['RegulationType'].append('Regulated')
				sqldict = bourange_same_length_array(sqldict)
				for rem in os.listdir(tempfolder):
					os.remove(os.path.join(tempfolder, rem)) 
				sleep(3)
			elif reg == regulatorName + ' 2':
				data_Excel=data_Excel.fillna('')
				data_Excel = data_Excel[~((data_Excel['Unnamed: 1'].str.strip() == '') & (data_Excel['Unnamed: 2'].str.strip() == ''))]
				data_Excel.reset_index(drop=True, inplace=True)
				data_Excel.columns = data_Excel.iloc[0]
				data_Excel.columns = list(map(str.strip, data_Excel))
				data_Excel = data_Excel[1:]
				sleep(3)
				for name,address,cngj,cep,email,website,phone in zip(data_Excel['NOME INSTITUIÇÃO'],
													data_Excel['ENDEREÇO'],
													data_Excel['CNPJ'],
													data_Excel['CEP'],
             										data_Excel['E-MAIL'],
             										data_Excel['SÍTIO NA INTERNET'],
             										data_Excel['FONE']                       
													):
					
					sqldict['Name'].append(name.strip())
					sqldict['InternalID_1'].append(str(cngj).strip())
					sqldict['InternalID_1_type'].append('CNPJ')
					sqldict['ListProcessDate'].append(processdate)
					sqldict['Address_1'].append(address.strip())
					sqldict['Zip'].append(cep.strip())
					sqldict['Email'].append(email.strip())
					sqldict['Website'].append(website.strip())
					sqldict['Phone'].append(phone)
					sqldict['ListName'].append(Typology[reg])
					sqldict['RegCtry'].append(reg.split(' ')[0]) 
					sqldict['RegCode'].append(reg.split(' ')[1])
					sqldict['ListCode'].append(reg.split(' ')[-1])          
					sqldict['RegulationType'].append('Regulated')
				sqldict = bourange_same_length_array(sqldict)
				print(reg)
				for rem in os.listdir(tempfolder):
					os.remove(os.path.join(tempfolder, rem)) 
				sleep(3)
			elif reg == regulatorName + ' 3':
				data_Excel=data_Excel.fillna('')
				data_Excel = data_Excel[~((data_Excel['Unnamed: 1'].str.strip() == '') & (data_Excel['Unnamed: 2'].str.strip() == ''))]
				data_Excel.reset_index(drop=True, inplace=True)
				data_Excel.columns = data_Excel.iloc[0]
				data_Excel.columns = list(map(str.strip, data_Excel))
				data_Excel = data_Excel[1:]
				sleep(3)				
				for name,address,cngj,cep,email,website,phone in zip(data_Excel['NOME INSTITUIÇÃO'],
													data_Excel['ENDEREÇO'],
													data_Excel['CNPJ'],
													data_Excel['CEP'],
             										data_Excel['E-MAIL'],
             										data_Excel['SÍTIO NA INTERNET'],
             										data_Excel['TELEFONE']                       
													):
					
					sqldict['Name'].append(name.strip())
					sqldict['InternalID_1'].append(str(cngj).strip())
					sqldict['InternalID_1_type'].append('CNPJ')
					sqldict['ListProcessDate'].append(processdate)
					sqldict['Address_1'].append(address.strip())
					sqldict['Zip'].append(cep.strip())
					sqldict['Email'].append(email.strip())
					sqldict['Website'].append(website.strip())
					sqldict['Phone'].append(phone)
					sqldict['ListName'].append(Typology[reg])
					sqldict['RegCtry'].append(reg.split(' ')[0]) 
					sqldict['RegCode'].append(reg.split(' ')[1])
					sqldict['ListCode'].append(reg.split(' ')[-1])          
					sqldict['RegulationType'].append('Regulated')
				sqldict = bourange_same_length_array(sqldict)
				print(reg)
				for rem in os.listdir(tempfolder):
					os.remove(os.path.join(tempfolder, rem)) 
				sleep(3)
			elif reg == regulatorName + ' 4':
				data_Excel=data_Excel.fillna('')
				data_Excel = data_Excel[~((data_Excel['Unnamed: 1'].str.strip() == '') & (data_Excel['Unnamed: 2'].str.strip() == ''))]
				data_Excel.reset_index(drop=True, inplace=True)
				data_Excel.columns = data_Excel.iloc[0]
				data_Excel.columns = list(map(str.strip, data_Excel))
				data_Excel = data_Excel[1:]
				sleep(3)				
				for name,address,cngj,cep,email,website,phone in zip(data_Excel['NOME INSTITUIÇÃO'],
													data_Excel['ENDEREÇO'],
													data_Excel['CNPJ'],
													data_Excel['CEP'],
             										data_Excel['E-MAIL'],
             										data_Excel['SITIO NA INTERNET'],
             										data_Excel['FONE']                       
													):
					
					sqldict['Name'].append(name.strip())
					sqldict['InternalID_1'].append(str(cngj).strip())
					sqldict['InternalID_1_type'].append('CNPJ')
					sqldict['ListProcessDate'].append(processdate)
					sqldict['Address_1'].append(address.strip())
					sqldict['Zip'].append(cep.strip())
					sqldict['Email'].append(email.strip())
					sqldict['Website'].append(website.strip())
					sqldict['Phone'].append(phone)
					sqldict['ListName'].append(Typology[reg])
					sqldict['RegCtry'].append(reg.split(' ')[0]) 
					sqldict['RegCode'].append(reg.split(' ')[1])
					sqldict['ListCode'].append(reg.split(' ')[-1])          
					sqldict['RegulationType'].append('Regulated')
				sqldict = bourange_same_length_array(sqldict)
				print(reg)
				for rem in os.listdir(tempfolder):
					os.remove(os.path.join(tempfolder, rem)) 
			elif reg == regulatorName + ' 5':
				data_Excel=data_Excel.fillna('')
				data_Excel = data_Excel[~((data_Excel['Unnamed: 1'].str.strip() == '') & (data_Excel['Unnamed: 2'].str.strip() == ''))]
				data_Excel.reset_index(drop=True, inplace=True)
				data_Excel.columns = data_Excel.iloc[0]
				data_Excel.columns = list(map(str.strip, data_Excel))
				data_Excel = data_Excel[1:]
				sleep(3)				
				for name,address,cngj,cep,email,website,phone in zip(data_Excel['NOME INSTITUIÇÃO'],
													data_Excel['ENDEREÇO'],
													data_Excel['CNPJ'],
													data_Excel['CEP'],
             										data_Excel['E-MAIL'],
             										data_Excel['SÍTIO NA INTERNET'],
             										data_Excel['TELEFONE']                       
													):
					
					sqldict['Name'].append(name.strip())
					sqldict['InternalID_1'].append(str(cngj).strip())
					sqldict['InternalID_1_type'].append('CNPJ')
					sqldict['ListProcessDate'].append(processdate)
					sqldict['Address_1'].append(address.strip())
					sqldict['Zip'].append(cep.strip())
					sqldict['Email'].append(email.strip())
					sqldict['Website'].append(website.strip())
					sqldict['Phone'].append(phone)
					sqldict['ListName'].append(Typology[reg])
					sqldict['RegCtry'].append(reg.split(' ')[0]) 
					sqldict['RegCode'].append(reg.split(' ')[1])
					sqldict['ListCode'].append(reg.split(' ')[-1])          
					sqldict['RegulationType'].append('Regulated')
				sqldict = bourange_same_length_array(sqldict)	
				print(reg)	
				for rem in os.listdir(tempfolder):
					os.remove(os.path.join(tempfolder, rem)) 
				sleep(3)	
    
	elif reg == 'BR BCB 8':
		sleep(10)
		soup = BeautifulSoup(driver.page_source, 'html.parser')
		sleep(5)
		pdf_files = soup.find('div',id='accordionParent-0').find('div',class_='card-body')
		sleep(5)
		file_lsits = pdf_files.find_all('div',class_='d-md-flex item-custom')
		sleep(5)
		for list in file_lsits:
			if 'spreadsheets' in list.text:
				driver.get(list.find('a')['href'])
		sleep(10)
		file = os.listdir(tempfolder)[0]
		sleep(3)
		filePath = os.path.join(tempfolder, file)							
		with zipfile.ZipFile(filePath, 'r') as zip_ref:
			sleep(3)
			zip_ref.extractall(tempfolder)
			print(f"[INFO] : - Zip extraction file = {zip_ref.filelist[0].filename}")
		sleep(3)
		extracted_files = os.listdir(tempfolder)
		sleep(3)
		xlsx_files = [file for file in extracted_files if file.endswith('.xlsx')]
		sleep(3)
		# Check if any .xlsx files were found
		if xlsx_files:
			print(f"[INFO] : - Found .xlsx file(s): {xlsx_files}")
		else:
			print("[ERROR] : - No .xlsx files found in the extracted contents")
		sleep(3)	
		filePath = os.path.join(tempfolder, xlsx_files[0])
		data_Excel=pd.read_excel(filePath)
		
		lang = ''
		try:
			filedf=pd.read_excel(filePath, sheet_name='quadro 10', header=None, dtype=str)
			lang = 'p'
		except:
			filedf=pd.read_excel(filePath, sheet_name='chart 10', header=None, dtype=str)
			lang = 'e'
		filedf.dropna(how='all', axis=1, inplace=True)

		filedf.dropna(subset= [2], inplace=True)
		filedf.fillna('', inplace=True)
		filedf.columns = filedf.iloc[0]
		filedf = filedf[1:]

		filedf.replace('\n',' ', regex=True, inplace=True)
		filedf.replace('\r',' ', regex=True, inplace=True)
		filedf.replace('\t',' ', regex=True, inplace=True)
		filedf.replace('  ',' ', regex=True, inplace=True)
		filedf.replace('NaN','', regex=True, inplace=True)

		if lang == 'p':
			for name,country,city in zip(filedf['Instituição Estrangeira'],filedf['País de Origem'],filedf['UF'],):
				print(name,translate_text(country),city)
				sqldict['Name'].append(name.strip())
				sqldict['ListProcessDate'].append(processdate)
				sqldict['Cntry'].append(translate_text(country))
				sqldict['City'].append(city.strip())
				sqldict['ListName'].append(Typology[reg])
				sqldict['RegCtry'].append(reg.split(' ')[0]) 
				sqldict['RegCode'].append(reg.split(' ')[1])
				sqldict['ListCode'].append(reg.split(' ')[-1])          
				sqldict['RegulationType'].append('Regulated')
			sqldict = bourange_same_length_array(sqldict)	
		elif lang == 'e':
			for name,country,city in zip(filedf['Foreign Institution'], filedf['Country of Origin '],filedf['UF'],):
				sqldict['Name'].append(name.strip())
				sqldict['ListProcessDate'].append(processdate)
				sqldict['Cntry'].append(country.strip())
				sqldict['City'].append(city.strip())
				sqldict['ListName'].append(Typology[reg])
				sqldict['RegCtry'].append(reg.split(' ')[0]) 
				sqldict['RegCode'].append(reg.split(' ')[1])
				sqldict['ListCode'].append(reg.split(' ')[-1])          
				sqldict['RegulationType'].append('Regulated')
			sqldict = bourange_same_length_array(sqldict)	
		for rem in os.listdir(tempfolder):
				os.remove(os.path.join(tempfolder, rem)) 
				sleep(3)	
				

				


Working with list BR BCB 1
----- Click Cookie-----
---- Work with conglomerado
This file downloads successfully!
[INFO] : - Found .xlsx file(s): ['202501CONGLOMERADO.xlsx']
---- Work with banco
This file downloads successfully!
[INFO] : - Found .xlsx file(s): ['202501BANCOS.xlsx']
BR BCB 2
---- Work with cooperativa
This file downloads successfully!
[INFO] : - Found .xlsx file(s): ['202501COOPERATIVAS.xlsx']
BR BCB 3
---- Work with sociedade
This file downloads successfully!
[INFO] : - Found .xlsx file(s): ['202501SOCIEDADES.xlsx']
BR BCB 4
---- Work with admConsorcio
This file downloads successfully!
[INFO] : - Found .xlsx file(s): ['202501ADMCONSORCIO.xlsx']
BR BCB 5
Working with list BR BCB 8
----- Click Cookie-----
[INFO] : - Zip extraction file = Quadros_Novembro de 2024.xlsx
[INFO] : - Found .xlsx file(s): ['Quadros_Novembro de 2024.xlsx']
COMMERZBANK AKTIENGESELLSCHAFT GERMANY SP
DZ BANK AG DEUTSCHE ZENTRAL-GENOSSENSCHAFTSBANK,FRANKFURT AM GERMANY SP
KREDITANSTALT FÜR WIEDERAUFB

In [12]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)



C:\Users\wuj1\AppData\Local\Temp\7\ipykernel_5192\2044201188.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [13]:
df.to_excel('BR_BCB_total.xlsx')

In [ ]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,99PAY INSTITUICAO DE PAGAMENTO S.A.,24313102,CNPJ PARTICIPANTE,,,...,,,99PAY IP - PRUDENCIAL,,,,,,,
1,,,,,,99PAY SOCIEDADE DE CRÉDITO DIRETO S.A.,,CNPJ PARTICIPANTE,,,...,,,99PAY IP - PRUDENCIAL,,,,,,,
2,,,,,,BANCO ABC BRASIL S.A.,28195667,CNPJ PARTICIPANTE,,,...,,,ABC-BRASIL - PRUDENCIAL,,,,,,,
3,,,,,,ABC BRASIL ADMINISTRACAO E PARTICIPACOES LTDA.,71995385,CNPJ PARTICIPANTE,,,...,,,ABC-BRASIL - PRUDENCIAL,,,,,,,
4,,,,,,ABC BRASIL COMERCIALIZADORA DE ENERGIA LTDA.,29198324,CNPJ PARTICIPANTE,,,...,,,ABC-BRASIL - PRUDENCIAL,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2814,,,,,,EFG BANK AG,,,,,...,,,,,,,,,,
2815,,,,,,MIRABAUD SCA,,,,,...,,,,,,,,,,
2816,,,,,,ZÜRCHER KANTONALBANK,,,,,...,,,,,,,,,,
2817,,,,,,BANCO SANTANDER S.A.,,,,,...,,,,,,,,,,
